# **DOWNLOAD DATASET FROM INGV VIA API PER BATCH**

In [1]:
import requests
import pandas as pd
from io import StringIO
import time
import os

# 1. Define the API and geographic parameters (Time will be handled dynamically)
url = "https://webservices.ingv.it/fdsnws/event/1/query"
base_params = {
    "minmag": 1.5,
    "minlat": 34.459,  
    "maxlat": 47.428,  
    "minlon": 3.867,   
    "maxlon": 21.27,  
    "format": "text"
}

# 2. Create monthly intervals from 1985 to 2025
# 'MS' stands for Month Start. We go up to 2026-01-01 to cover all of Dec 2025
time_bins = pd.date_range(start='1985-01-01', end='2026-01-01', freq='MS')

all_earthquakes = [] # List to store dataframes of each month
total_batches = len(time_bins) - 1

print(f"Starting batch download ({total_batches} monthly batches)...")
print("This will take a few minutes. Please don't interrupt the process.")

# 3. Loop through each time interval
for i in range(total_batches):
    start_time = time_bins[i].strftime("%Y-%m-%dT00:00:00")
    # Subtract 1 second from the next month's start to get the end of the current month
    end_time = (time_bins[i+1] - pd.Timedelta(seconds=1)).strftime("%Y-%m-%dT23:59:59")
    
    # Update parameters for the current batch
    current_params = base_params.copy()
    current_params["starttime"] = start_time
    current_params["endtime"] = end_time
    
    # Make the request
    response = requests.get(url, params=current_params)
    
    # Handle the response
    if response.status_code == 200:
        # Data found: read and append to list
        data_io = StringIO(response.text)
        df_batch = pd.read_csv(data_io, sep='|')
        all_earthquakes.append(df_batch)
    
    elif response.status_code == 204:
        # 204 No Content: perfectly normal if no earthquakes >= 1.5 occurred that month
        pass
        
    else:
        # Log any other errors but keep trying next batches
        print(f"\nError in batch {start_time} to {end_time}: HTTP {response.status_code}")
    
    # Print progress every 24 months (2 years) to keep the output clean
    if (i + 1) % 24 == 0 or (i + 1) == total_batches:
        print(f"Progress: {i + 1}/{total_batches} batches processed...")
        
    # Crucial: sleep for a fraction of a second to be polite to the INGV servers
    time.sleep(0.3)

# 4. Merge all monthly DataFrames into one massive dataset
print("\nMerging all batches into a single DataFrame...")
df_raw = pd.concat(all_earthquakes, ignore_index=True)

# Standardize column names
df_raw.columns = [col.lower() for col in df_raw.columns]

# 5. Save the final comprehensive dataset
filename = "data/INGV/ingv_raw_earthquakes_1985_2025_complete.csv"
df_raw.to_csv(filename, index=False)

file_path = os.path.abspath(filename)
print(f"Success! {len(df_raw)} total earthquakes retrieved and merged.")
print(f"Complete dataset saved to: {file_path}")

# Display the final head
display(df_raw.head())

Starting batch download (492 monthly batches)...
This will take a few minutes. Please don't interrupt the process.
Progress: 24/492 batches processed...
Progress: 48/492 batches processed...
Progress: 72/492 batches processed...
Progress: 96/492 batches processed...
Progress: 120/492 batches processed...
Progress: 144/492 batches processed...
Progress: 168/492 batches processed...
Progress: 192/492 batches processed...
Progress: 216/492 batches processed...
Progress: 240/492 batches processed...
Progress: 264/492 batches processed...
Progress: 288/492 batches processed...
Progress: 312/492 batches processed...
Progress: 336/492 batches processed...
Progress: 360/492 batches processed...
Progress: 384/492 batches processed...
Progress: 408/492 batches processed...
Progress: 432/492 batches processed...
Progress: 456/492 batches processed...
Progress: 480/492 batches processed...
Progress: 492/492 batches processed...

Merging all batches into a single DataFrame...
Success! 216097 total 

,#eventid,time,latitude,longitude,depth/km,author,catalog,contributor,contributorid,magtype,magnitude,magauthor,eventlocationname,eventtype
0,2289,1985-01-28T08:45:53.200000,42.515,13.313,10.0,BULLETIN-VAX,NaN,NaN,NaN,Md,2.5,--,1 km SE Capitignano (AQ),earthquake
1,2249,1985-01-27T03:24:43.880000,40.759,15.250,10.0,BULLETIN-VAX,NaN,NaN,NaN,Md,2.9,--,3 km NW Valva (SA),earthquake
2,2159,1985-01-25T23:33:12.210000,39.135,16.000,99.7,BULLETIN-VAX,NaN,NaN,NaN,Md,3.1,--,Costa Calabra nord-occidentale (Cosenza),earthquake
3,2149,1985-01-25T22:26:11.110000,40.571,19.310,10.0,BULLETIN-VAX,NaN,NaN,NaN,Md,2.8,--,Costa Albanese settentrionale (ALBANIA),earthquake
4,2089,1985-01-23T16:11:34.960000,38.170,20.619,10.0,BULLETIN-VAX,NaN,NaN,NaN,M,4.3,--,Costa Greca Ionica (GRECIA),earthquake


# ***Check of DB***

In [3]:
print(f"The uploaded dataframe has {len(df_raw):_} rows and {len(df_raw.columns):_} columns.")
memory = df_raw.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"It occupies {memory:_.2f} MB in RAM memory")
print(df_raw.columns)
display( pd.concat([df_raw.head(5),df_raw.tail(5)]) )

The uploaded dataframe has 216_097 rows and 14 columns.
It occupies 89.80 MB in RAM memory
Index(['#eventid', 'time', 'latitude', 'longitude', 'depth/km', 'author',
       'catalog', 'contributor', 'contributorid', 'magtype', 'magnitude',
       'magauthor', 'eventlocationname', 'eventtype'],
      dtype='str')


,#eventid,time,latitude,longitude,depth/km,author,catalog,contributor,contributorid,magtype,magnitude,magauthor,eventlocationname,eventtype
0,2289,1985-01-28T08:45:53.200000,42.5150,13.3130,10.0,BULLETIN-VAX,NaN,NaN,NaN,Md,2.5,--,1 km SE Capitignano (AQ),earthquake
1,2249,1985-01-27T03:24:43.880000,40.7590,15.2500,10.0,BULLETIN-VAX,NaN,NaN,NaN,Md,2.9,--,3 km NW Valva (SA),earthquake
2,2159,1985-01-25T23:33:12.210000,39.1350,16.0000,99.7,BULLETIN-VAX,NaN,NaN,NaN,Md,3.1,--,Costa Calabra nord-occidentale (Cosenza),earthquake
3,2149,1985-01-25T22:26:11.110000,40.5710,19.3100,10.0,BULLETIN-VAX,NaN,NaN,NaN,Md,2.8,--,Costa Albanese settentrionale (ALBANIA),earthquake
4,2089,1985-01-23T16:11:34.960000,38.1700,20.6190,10.0,BULLETIN-VAX,NaN,NaN,NaN,M,4.3,--,Costa Greca Ionica (GRECIA),earthquake
216092,44752202,2025-12-01T10:35:34.450000,45.8927,11.9652,7.8,SURVEY-INGV,NaN,NaN,NaN,ML,1.5,--,3 km W Valdobbiadene (TV),earthquake
216093,44751392,2025-12-01T07:12:13.000000,42.7787,12.7258,10.4,SURVEY-INGV,NaN,NaN,NaN,ML,1.9,--,5 km N Spoleto (PG),earthquake
216094,44750742,2025-12-01T03:10:19.730000,45.8738,7.0510,11.1,SURVEY-INGV,NaN,NaN,NaN,ML,1.6,--,11 km NE Courmayeur (AO),earthquake
216095,44750662,2025-12-01T02:25:48.990000,44.1677,9.9072,6.4,SURVEY-INGV,NaN,NaN,NaN,ML,1.7,--,1 km NW Santo Stefano di Magra (SP),earthquake
216096,44750592,2025-12-01T01:27:50.160000,44.5130,10.2288,25.0,SURVEY-INGV,NaN,NaN,NaN,ML,2.4,--,2 km E Tizzano Val Parma (PR),earthquake


# ***OPTIONAL ANALYSIS FILTER ONLY ITALY EARTH DATA***

In [4]:
import pandas as pd
import geopandas as gpd
import warnings

# Suppress simple warnings for a cleaner output in your notebook

# 1. Load the raw dataset
print("Loading earthquake data...")
df = pd.read_csv("data/INGV/ingv_raw_earthquakes_1985_2025_complete.csv")

# 2. Convert the DataFrame into a GeoDataFrame
# EPSG:4326 is the standard coordinate system for GPS (Latitude and Longitude)
gdf = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326" 
)

# 3. Fetch world map directly from the web 
# This bypasses the deprecated geopandas.datasets module by downloading the zip directly
print("Fetching geographical boundaries from Natural Earth...")
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

# 4. Isolate Italy and perform the "Point in Polygon" check
italy_boundary = world[world['ADMIN'] == "Italy"]

print("Filtering points (this might take a few seconds)...")
is_in_italy = gdf.within(italy_boundary.unary_union)

# 5. Filter the original dataframe
df_italy = df[is_in_italy].copy()

print(f"\n--- Filtering Results ---")
print(f"Original events: {len(df)}")
print(f"Events strictly within Italy: {len(df_italy)}")
print(f"Events removed (neighboring countries/open sea): {len(df) - len(df_italy)}")

# Save the filtered version
df_italy.to_csv("data/INGV/ingv_earth_italy_only_1985_2025.csv", index=False)
print("\nFiltered dataset saved successfully to 'data/INGV/ingv_earth_italy_only_1985_2025.csv'!")

Loading earthquake data...
Fetching geographical boundaries from Natural Earth...
Filtering points (this might take a few seconds)...


/tmp/ipykernel_7627/1743755501.py:29: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  is_in_italy = gdf.within(italy_boundary.unary_union)



--- Filtering Results ---
Original events: 216097
Events strictly within Italy: 183324
Events removed (neighboring countries/open sea): 32773

Filtered dataset saved successfully to 'data/INGV/ingv_earth_italy_only_1985_2025.csv'!


# ***OPTIONAL ANALISYS FILTER ONLY ITALY EARTH AND SEA DATA***

In [5]:
import pandas as pd
import geopandas as gpd
import warnings

warnings.filterwarnings('ignore')

print("Loading earthquake data...")
df = pd.read_csv("data/INGV/ingv_raw_earthquakes_1985_2025_complete.csv")

# 1. Create GeoDataFrame (EPSG:4326 is Lat/Lon in degrees)
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326"
)

print("Fetching geographical boundaries...")
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)
italy_land = world[world['ADMIN'] == "Italy"]

# --- THE MAGIC FOR THE SEA HAPPENS HERE ---

# 2. Project Italy to a Metric System (UTM Zone 32N - EPSG:32632 is perfect for Italy)
# This converts our map from degrees to meters so we can measure exact distances
italy_metric = italy_land.to_crs(epsg=32632)

# 3. Apply the Buffer to include the sea
# 12 Nautical Miles = approx 22,224 meters. 
# We'll use 30,000 meters (30 km) just to be safe and catch coastal fault lines
BUFFER_METERS = 30000 
print(f"Expanding borders by {BUFFER_METERS/1000} km to include territorial waters...")
italy_with_sea_metric = italy_metric.buffer(BUFFER_METERS)

# 4. Project back to standard GPS coordinates (Degrees) to match our earthquake points
italy_with_sea = italy_with_sea_metric.to_crs(epsg=4326)

# -------------------------------------------

print("Filtering points (checking Land + Sea)...")
# Check points against the new, expanded polygon
is_in_italy_and_sea = gdf.within(italy_with_sea.unary_union)

# 5. Filter the dataframe
df_italy_sea = df[is_in_italy_and_sea].copy()

print(f"\n--- Filtering Results ---")
print(f"Original events: {len(df)}")
print(f"Events in Italy (Land + 30km Sea): {len(df_italy_sea)}")
print(f"Events removed (far away): {len(df) - len(df_italy_sea)}")

# Save this definitively!
df_italy_sea.to_csv("data/INGV/ingv_italy_and_sea_1985_2025.csv", index=False)
print("\nFinal dataset saved successfully!")

Loading earthquake data...
Fetching geographical boundaries...
Expanding borders by 30.0 km to include territorial waters...
Filtering points (checking Land + Sea)...

--- Filtering Results ---
Original events: 216097
Events in Italy (Land + 30km Sea): 198909
Events removed (far away): 17188

Final dataset saved successfully!
